### 1. What was the average time each driver spent at the pit stop for each race?  
### Provide also the slowest and fastest pit stop in each race.


In [0]:
# Import all libraries
from pyspark.sql.functions import (
    avg,min, max, col, round,sum, when, translate, regexp_replace, split, size, trim,
    upper, substring, concat, element_at, length,to_date,floor, datediff)

from pyspark.sql.window import Window

In [0]:
# Import Dataset pitstops 
df_pitstops = spark.read.csv('/Volumes/gr5069/raw/f1_data/pit_stops.csv',header=True)
display(df_pitstops)

#### logic for Q1:
By observing the dataset, for each race with each driver, there are multiple stops. The variable "stop" only show the order of the stop for a driver within a race. The variable "milliseconds show how long that stop lasted. To find the average time each driver spent at the pit stop in each race, we can group data by "raceId" and "driverId", placing together all pit stop records for the same driver in the same race. Then, wew can calculate the mean of "milliseconds" with each group, which gives us the average time for each driver in each race. 

Moreover, to find the fastest and slowest pit stop in each race, we can group the data by "raceId" to make groups for each race. Then, we can find min and max values of "milliseconds" for each race. 

In [0]:
# Convert milliseconds to integer
df_pitstops = df_pitstops.withColumn("milliseconds", col("milliseconds").cast("int"))

# Average pit stop time for each driver in each race
driver_avg_pitstop = df_pitstops.groupBy("raceId", "driverId").agg(
    avg("milliseconds").alias("avg_pitstop_ms"))

display(driver_avg_pitstop)


In [0]:
# Fastest and slowest pit stop in each race
race_extremes = df_pitstops.groupBy('raceId').agg(
    min('milliseconds').alias('fastest_pitstop_per_race'),
    max('milliseconds').alias('slowest_pitstop_per_race'))

display(race_extremes)

#### Explain codes:
The first step converts "milliseconds" datatype from String to Integer. We need it to be number to apply functions like averages, min, and max. `col("milliseconds")` selects the "millisecond" column, and `cast("int")` converts it from a string type into an integer type.  `withColumn()` replaces the original column with its integer version.

The next step, we use `groupBy("raceId", "driverId")` to group all rows that have same race id and driver id. So, all pitstops for one driver in one race are put together into one group. `agg()` apply summary calculation to each group, and `avg("milliseconds")` adds all pit stop times in each group and divides by the number of rows in that group. Thus, we get averages for each driver within each race. `alias("avg_pitstop_ms")` renames the result col.

Then, we try to get the fastest and slowest pit stop in each race. `groupBy("raceId")` groups all pit stop records for same race.`min("milliseconds")` finds the smallest pit stop time within each race, representing the fastest pit stop. `max("milliseconds")` finds the largest pit stop time for each race, representing the slowest pit stop. 

### 2. Rank order by finishing position the average time spent at the pit stop in each race.

#### Logic for Q2：
For this question, we need to rank driver's average pit stop time by finishing position in each race. We already have the average pit stop time from previous question.

We will use the "results" datasets to obtain each driveres' finishing position. The dataset contains variable such as "position", "positionText", "positionOrder", and "rank", but the most approiate variavle will be "positionOrder." "positionOrder" provide the final race order with only numbers. By contrast, "position" and "positionText" may contain non-numberic or missing values. And "rank" is related to fastest lap time, not the finishing position. 

After that, we will keep only the variables needed for this task: "raceId", "driverId", and "positionOrder". Then, we will join the finishing position information with average pit stop summary using "raceId" and "driverId", so that each row contains both drivers' finishing position and average pit stop time for same race. 

Finally, we will sort the result by "raceId" and "positionOrder", so we have drivers ranked by finishing position within each race, together with their average pit stop time.

In [0]:
#Import dataset results
df_results = spark.read.csv('/Volumes/gr5069/raw/f1_data/results.csv',header=True)
display(df_results)

In [0]:
# Convert finishing position to integer data type
df_results = df_results.withColumn("positionOrder", col("positionOrder").cast("int"))

# Select finishing position from results
finish_position = df_results.select("raceId","driverId","positionOrder")

'''
Use three different join methods to join finishing position with average pit stop time. 
Each serve for differnt purpose.
1. Left join: It's used when main goal is to preserve the complete race result order, even some some drivers or races do not have matching pit stop records. For those unmatched cases, the average pit stop time will appear as `null`. 
2. Inner join: Only drivers who have both a finishing position and a recorded average pit stop time are included. It'used when the focus is on comparing finishing position with average pit stop time.
3. Right join: It's used when the priority is to preserve all observations that contain pit stop data.
'''
# Left join:
finishPosition_avgPitstop = finish_position.join(driver_avg_pitstop,
                                                    on=["raceId", "driverId"],how="left")

# Sort by race and finishing position
display(finishPosition_avgPitstop.orderBy("raceId", "positionOrder"))

In [0]:
# Inner join:
finishPosition_avgPitstop_inner = finish_position.join(driver_avg_pitstop,
                                                       on=["raceId", "driverId"],how="inner")

# Sort by race and finishing position
display(finishPosition_avgPitstop_inner.orderBy("raceId", "positionOrder"))

In [0]:
# Right join:
finishPosition_avgPitstop_right = finish_position.join(driver_avg_pitstop,
                                                    on=["raceId", "driverId"],how="right")

# Sort by race and finishing position
display(finishPosition_avgPitstop_right.orderBy("raceId", "positionOrder"))

#### Explain codes:
After importing the dataset "results", we firstly convert the variable `positionOrder` into integer type using function `cast(int)`. `withColumn()` replaces the existing `positionOrder` column with its integer version.

Next, we create a table containing only the `raceId`, `driverId`, and `positionOrder` because we want to use `raceId` and `driverId` as keys put `positionOrder` together with average pit stop time data. 

Then, we join the finish position table with the `driver_avg_pitstop` table. In all three join methods we used, the matching keys are: `on=["raceId", "driverId"]`. This matches rows where both race id and driver id are the same in two data.

The first join is a left join, keeping all rows from `finish_position` table. All finishing positions from the results dataset are keeped, even if some rows have no matching pit stop information. `null` represents no corresponding pit stop record. This method is useful when the priority is to preserve the full finishing order for every race.

The second is inner join, keeping only drivers who have both a finishing position and an average pit stop value. It's useful when the focus is on comparing finishing position with average pit stop time.

The third is right join, keeping all rows from `driver_avg_pitstop`. All pit stop data are keeped, even if a matching finishing position is not found.

After each join, the code uses `orderBy("raceId", "positionOrder")`, which sorts the output first by race and then by finishing position. So we have results that are ordered by finishing position within each race.

### 3.  Insert the missing code (e.g: ALO for Alonso) for drivers based on the 'drivers' dataset. Explain your logic.


#### Logic for Question 3
To answer this question, we will first extract rows from driver data where the `code` value is missing. In this dataset, missing codes appear as `\N`, but we also include `null` to prevent exceptions.

First, we will clean the surname values, normalizing accented characters so that names with diacritics can be handled consistently. Then, we remove special symbols but keep spaces, since drivers may have surnames that contains multiple words. 

In addition, the code counts how many surname parts each driver has and apply differnt rules based on the number of surname parts.

- If the surname has **one part**, we will take the first three letters of the surname and convert them to uppercase. If the surname has fewer than three letters, we use the available surname letters and append the first letter of the forename.

- If the surname has **two parts**, we will take the first letter of each surname part and then append the first letter of the forename. The result is converted to uppercase.

- If the surname has **three and more parts**, we will take the first letter of first three surname parts and convert the result to uppercase.

This logic is consistent with the naming patterns observed in the dataset, while preserving all original non-missing codes.

In [0]:
# Import drivers dataset
df_drivers = spark.read.csv('/Volumes/gr5069/raw/f1_data/drivers.csv',header=True)
display(df_drivers)

In [0]:
from pyspark.sql.functions import (
    col, when, translate, regexp_replace, split, size, trim,
    upper, substring, concat, element_at, length)

# Keep only rows with missing code
missing_code = df_drivers.filter((col("code") == "\\N") | (col("code").isNull()))

# Normalize surname by translating accented characters
normalized_surname = translate(col("surname"),
                               "äöüàáâãåæçèéêëìíîïñòóôõøùúûüýÿÄÖÜÀÁÂÃÅÆÇÈÉÊËÌÍÎÏÑÒÓÔÕØÙÚÛÜÝ","aouaaaaaceeeeiiiinooooouuuuyyAOUAAAAACEEEEIIIINOOOOOUUUUY")

# Remove special characters but keep spaces
missing_code = missing_code.withColumn("clean_surname",
                                       trim(regexp_replace(normalized_surname, "[^A-Za-z ]", "")))

# Count number of surname parts
missing_code = missing_code.withColumn("surname_part_count",
                                       size(split(col("clean_surname"), " ")))

# Show all missing-code rows with surname type
display(
    missing_code.select("driverId", "forename", "surname", "clean_surname", "surname_part_count")
)

In [0]:
# Count single vs multiple surname cases
display(missing_code.groupBy("surname_part_count").count())

In [0]:
# Split surname into parts
missing_code = missing_code.withColumn(
    "surname_parts",
    split(col("clean_surname"), " "))

# Generate code for one-part surnames
missing_code = missing_code.withColumn(
    "code_part1",
    when(col("surname_part_count") == 1,
         upper(
             when(
                 length(col("clean_surname")) >= 3,
                 substring(col("clean_surname"), 1, 3)).otherwise(
                     concat(col("clean_surname"), substring(col("forename"), 1, 1)))
            )
        )
    )

# Generate code for two-part surnames
missing_code = missing_code.withColumn(
    "code_part2",
    when(col("surname_part_count") == 2,
         upper(
             concat(
                 substring(element_at(col("surname_parts"), 1), 1, 1),
                 substring(element_at(col("surname_parts"), 2), 1, 1),
                 substring(col("forename"), 1, 1))
             )
         )
    )

#Generate code for three and more parts surnames
missing_code = missing_code.withColumn(
    "code_part3",
    when(col("surname_part_count") >= 3,
         upper(
             concat(
                 substring(element_at(col("surname_parts"), 1), 1, 1),
                 substring(element_at(col("surname_parts"), 2), 1, 1),
                 substring(element_at(col("surname_parts"), 3), 1, 1))
             )
         )
    )

In [0]:
# Combine all generated code into one final code column
missing_code = missing_code.withColumn(
    "new_code",
    when(col("surname_part_count") == 1, col("code_part1"))
    .when(col("surname_part_count") == 2, col("code_part2"))
    .when(col("surname_part_count") >= 3, col("code_part3")))

display(missing_code)

In [0]:
# Keep only the generated code for drivers in missing code
filled_codes = missing_code.select("driverId", "new_code")

# Join generated codes back to the original drivers table
df_drivers_codes = df_drivers.join(
    filled_codes,
    on="driverId",
    how="left")

# Replace missing code
df_drivers_codes = df_drivers_codes.withColumn(
    "all_code",
    when(
        (col("code") == "\\N") | (col("code").isNull()),
        col("new_code")
        ).otherwise(col("code"))
    )

# Drop new_code and keep final_code only
df_drivers_codes = df_drivers_codes.drop("new_code")

display(df_drivers_codes)

#### Explain codes:
To begin with, we firstly created the new DataFrame `missing_code` using `df_drivers.filter((col("code") == "\\N") | (col("code").isNull()))`. By doing this, `filter()` keeps only the rows where the `code` value is `\\N` or `null`. So, we will only make changes on missing codes, while the existing valid codes will remain unchanged. 

Next, we normalizes drivers' surnames with: `translate(col("surname"), ...)`, replacing accented characters with standard alphabetic characters. This step makes names to be simpler form that can be processed consistently.

Then, we create a cleaned surname column using `withColumn("clean_surname", trim(regexp_replace(normalized_surname, "[^A-Za-z ]", "")))`. Here, `withColumn()` creates a new column `clean_surname`. `regexp_replace()` removes all characters that are not letters or spaces, so punctuation and special symbols are dropped. `[^A-Za-z ]` represents anything that is not an uppercase letter, lowercase letter, or a space. `trim()` removes leading and trailing spaces. This keeps the spaces between different parts of names, so we can have a standardized surname string that can be split into parts.

After this, we count how many surname parts each driver has using `size(split(col("clean_surname"), " "))`. `split()` separates `clean_surname` into an array using spaces as separators.  `size()` counts how many elements are in that array. This allows us to indentify whetheer a surname has one, two, or three parts, which determines which rule will be used to generate the driver code. 

`missing_code.groupBy("surname_part_count").count()` groups rows by number of suranmes parts, and `count()` counts how many rows fall into each group. This serve as a summary to double check how many different cases (how many parts in surname) we have. Next, `withColumn("surname_parts", split(col("clean_surname"), " "))` stores the split surname as an array in column `surname_parts`. This array is used later when the code needs to access the first, second, or third surname part individually.

Moreover, the first generation rule is applied to one part surnames:   `withColumn("code_part1", when(col("surname_part_count") == 1, ...))`. `when()` is a conditional rule. If `surname_part_count` is 1, the code generates a value for `code_part1`. Then, we checks whether the cleaned surname has at least three letters using `length(col("clean_surname")) >= 3`. If it does, `substring(col("clean_surname"), 1, 3)` takes the first three letters of the surname. If not, `concat(col("clean_surname"), substring(col("forename"), 1, 1))` appends the first letter of the forename to the short surname. `upper()` converts the generated code to uppercase.

The second generation rule handles two parts surnames:  
If a surname has two parts, `element_at(col("surname_parts"), 1)` and `element_at(col("surname_parts"), 2)` access the first and second surname parts. `substring(..., 1, 1)` extracts the first letter of each part. Then the code also takes the first letter of the forename with `substring(col("forename"), 1, 1)`. The function `concat()` combines these three letters, and `upper()` converts the result to uppercase.

The third rule handles three and more parts surnames:
In this case, the code uses `element_at()` to access the first, second, and third surname parts. It extracts the first character from each part using `substring(..., 1, 1)`, combines them using `concat()`, and then uses `upper()` to produce three letter uppercase code.

After the three separate code chunks are created, we combine them into one final generated code column:
`withColumn("new_code", when(col("surname_part_count") == 1, col("code_part1")).when(col("surname_part_count") == 2, col("code_part2")).when(col("surname_part_count") == 3, col("code_part3")))`. This step uses chained `when()` conditions to assign the correct generated code depending on the number of surname parts. If the surname has one part, `new_code` takes the value from `code_part1`, etc. 

Next, the code creates a smaller table `filled_codes` with `missing_code.select("driverId", "new_code")`. This keeps only `driverId` and the newly generated code. And we join the generated codes back to the original table: `df_drivers.join(filled_codes, on="driverId", how="left")`. The argument `how="left"` means that all rows from the original `df_drivers` table are preserved, and matching generated codes are attached when available.

After the join, the code creates the final replacement column:
`withColumn("all_code", when((col("code") == "\\N") | (col("code").isNull()), col("new_code")).otherwise(col("code")))`. This checks whether the original `code` is missing. If it is missing, the new generated code is inserted. If the original drivers' code already exists, the code keeps that original value. The function `otherwise()` ensures that valid existing codes are preserved.

Finally, `df_drivers_codes = df_drivers_codes.drop("new_code")` removes the temporary `new_code` column from the final dataset, so we have a cleaner dataset.

4. Who is the youngest and the oldest driver in each race? Create a new column called “Age”.   
Explain your definition of "age".

#### Logic for Q4:
First, the variable **Age** is defined as the drivers' age in whole years on the date of the race. So, we compare the drivers' date of birth with the race date, calculating how many full years old the driver was at that race. 

We will join the `races` dataset to bring in the race date and the `drivers` dataset to bring in each driver’s name and date of birth to `results` dataset.vAfter combining these datasets, we will convert the date columns into date format. Next, we will create `Age` column by calculating the difference between the race date and the driver’s date of birth, converting that difference into years. This gives drivers' age at the time of the race.

After having `Age` column, we will group the data by `raceId` and calculate the minimum and maximum age in each race. Finally, we will list the youngest and oldest summary seperately with drivers' names. 

In [0]:
# Import dataset races
df_races= spark.read.csv('/Volumes/gr5069/raw/f1_data/races.csv',header=True)
display(df_races)

In [0]:
# Join race date and driver information into results
df_results_age = df_results.join(
    df_races.select("raceId", "date"),
    on="raceId",
    how="left"
).join(
    df_drivers.select("driverId", "forename", "surname", "dob"),
    on="driverId",
    how="left")

display(df_results_age)

In [0]:
# Convert date columns to proper date format
df_results_age = df_results_age.withColumn(
    "race_date", to_date(col("date"), "yyyy-MM-dd")
).withColumn(
    "dob", to_date(col("dob"), "yyyy-MM-dd"))

# Create Age column
df_results_age = df_results_age.withColumn(
    "Age",
    floor(datediff(col("race_date"), col("dob")) / 365))

display(df_results_age)

In [0]:
# Youngest and oldest age in each race
results_age_summary=df_results_age.groupBy("raceId").agg(
    min("Age").alias("youngest_age"),
    max("Age").alias("oldest_age"))

display(results_age_summary)

In [0]:
# Youngest drivers in each race
youngest_drivers = df_results_age.join(
    results_age_summary,
    on="raceId",
    how="inner"
).filter(
    col("Age") == col("youngest_age")
).select(
    "raceId",
    "forename",
    "surname",
    "Age")

display(youngest_drivers)

In [0]:
# Oldest driver in each race
oldest_drivers = df_results_age.join(
    results_age_summary,
    on="raceId",
    how="inner"
).filter(
    col("Age") == col("oldest_age")
).select(
    "raceId",
    "forename",
    "surname",
    "Age")

display(oldest_drivers)

#### Code Explain:
First, we join race information and driver information into the `results` dataset. The `results` dataset is used as the base table because it identifies which drivers participated in which races. 

After the joins, the code converts the string date variables into date format:
`withColumn("race_date", to_date(col("date"), "yyyy-MM-dd")).withColumn("dob", to_date(col("dob"), "yyyy-MM-dd"))`. `col("date")` selects the race date column, and `col("dob")` selects the date-of-birth column. `to_date()` converts these string values into date type using the format `"yyyy-MM-dd"`. 

Then, we create the `Age` column with:
`withColumn("Age", floor(datediff(col("race_date"), col("dob")) / 365))`. `datediff()` calculates the number of days between the race date and the drivers' date of birth. Dividing this value by `365` converts the difference from days into approximate years. `floor()` rounds the result down to the nearest whole number. So, age is defined here as the drivers' age in full completed years on race day.

Next, we identify the minimum and maximum age in each race:
`df_results_age.groupBy("raceId").agg(min("Age").alias("youngest_age"), max("Age").alias("oldest_age"))`. `groupBy("raceId")` groups all rows from the same race together.`min("Age")` finds the youngest age in that race, and `max("Age")` finds the oldest age in that race. The function `alias()` renames these summary columns as `youngest_age` and `oldest_age`.

After this, we join the summary back to table that has drivers info: `df_results_age.join(results_age_summary, on="raceId", how="inner")`. `inner` join keeps only rows where the `raceId` appears in both tables. This makes it possible to compare each driver’s age to the minimum and maximum age for that same race.

To find the youngest drivers, we use: `filter(col("Age") == col("youngest_age"))`, keeping only the rows where the driver’s age is equal to the youngest age in that race. The same logic is used to find the oldest drivers.

5. At any given race, how many podiums does each driver have? create three new columns to show -   
on any given race - the number of wins, the number of 2nd places, and the number of 3rd places for each driver

#### Logic for Q5:
For each driver and each race, we will calculate three cumulative counts: the number of wins, the number of 2nd place finishes, and the number of 3rd place finishes.

We will begin with the `results_age` dataset because it contains the race result for each driver, drivers' information, race date. For each row, we will create indicator variables that show whether the driver finished 1st, 2nd, or 3rd in that race.

After creating indicator columns, we will use window function partitioned by `driverId` and ordered by race date. This allows us to calculate cumulative sums for each driver across time. The cumulative sum of the 1st place indicator gives the total number of wins up to that race. The same method is used for 2nd place and 3rd place finishes.

In [0]:
# Create indicator columns for 1st, 2nd, and 3rd place finishes
df_results_age = df_results_age.withColumn(
    "first_place",
    when(col("positionOrder") == "1", 1).otherwise(0)
).withColumn(
    "second_place",
    when(col("positionOrder") == "2", 1).otherwise(0)
).withColumn(
    "third_place",
    when(col("positionOrder") == "3", 1).otherwise(0)
    )

display(df_results_age)

In [0]:
# Define window by driver and race date
podium_window = Window.partitionBy("driverId").orderBy("race_date") \
                      .rowsBetween(Window.unboundedPreceding, Window.currentRow)

# Create cumulative columns
df_results_age = df_results_age.withColumn(
    "first_so_far",
    sum("first_place").over(podium_window)
).withColumn(
    "seconds_so_far",
    sum("second_place").over(podium_window)
).withColumn(
    "thirds_so_far",
    sum("third_place").over(podium_window)
    )

display(
    df_results_age.select(
        "raceId","driverId", "forename","surname","race_date",
        "positionOrder", "first_so_far","seconds_so_far","thirds_so_far"
        ).orderBy( "race_date")
    )

#### Code Explain:
The code first creates three indicator columns called `first_place`, `second_place`, and `third_place`. These columns identify whether a driver finished 1st, 2nd, or 3rd place in a given race. For example, `when(col("positionOrder") == "1", 1).otherwise(0)` assigns a value of 1 if the driver finished in 1st place and 0 otherwise. The same logic is used for 2nd and 3rd place finishes. Those indicator columns can help us to get the sum of podium numbers by adding the numebrs.

Next, the code defines a window called `podium_window` with: `Window.partitionBy("driverId").orderBy("race_date").rowsBetween(Window.unboundedPreceding, Window.currentRow)`. In PySpark, a Window function performs calculations across a set of rows (a "window") that are related to the current row. `partitionBy("driverId")` means that the cumulative calculations are done separately for each driver. `orderBy("race_date")` arranges each driver’s races in chronological order, so the cumulative counts follow the order in which the races happened. The function `rowsBetween(Window.unboundedPreceding, Window.currentRow)` tells Spark to include all rows from the driver’s first recorded race up to the current race row. This creates a cumulative window.

After defining the window, the code creates three cumulative columns:`first_so_far`, `seconds_so_far`, and `thirds_so_far`. `sum("first_place").over(podium_window)` adds up all the values in the `first_place` column for each driver from their earliest race up to the current race. Since `first_place` contains only 1s and 0s, this running sum gives the total number of wins that driver has accumulated by that race. The same logic is used for `second_place` and `third_place`, which produce cumulative counts of 2nd place and 3rd place finishes.

6. Continue exploring the data by answering your own question.

My question is: At what age did each driver get their first podium or first win? 

#### Logic:
We will explore at what age each driver earned their first podium and their first win, and then identify which driver was the youngest when achieving a first win.

The first podium is defined as the first race in which a driver finished in 1st, 2nd, or 3rd place. First win is the first race in which a driver finished in 1st place. Age is defined in the same way as before: the driver’s age in whole years on the date of the race.

To answer this question, we will use the `df_results_age` table. First, we will filter the rows to identify podium finishes and winning finishes. Then, for each driver, we will find the earliest race date on which they achieved a podium and the earliest race date on which they achieved a win.

After identifying those first dates, we join them back to the driver-level results table in order to recover the corresponding age and driver name. This gives one table for each driver’s first podium age and another table for each driver’s first win age.

In [0]:
# First podium date
first_podium_date = df_results_age.filter(
    (col("positionOrder") == "1") |
    (col("positionOrder") == "2") |
    (col("positionOrder") == "3")
).groupBy("driverId").agg(
    min("race_date").alias("first_podium_date"))

# First win date
first_win_date = df_results_age.filter(
    col("positionOrder") == "1"
).groupBy("driverId").agg(
    min("race_date").alias("first_win_date"))

In [0]:
# Get age and driver name for first podium
first_podium = df_results_age.join(
    first_podium_date,
    on="driverId",
    how="inner"
).filter(
    col("race_date") == col("first_podium_date")
).select(
    "driverId","forename",
    "surname","date","Age"
).withColumnRenamed("date", "first_podium_date") \
 .withColumnRenamed("Age", "first_podium_age")

# Show first podium age for each driver
display(first_podium.orderBy("first_podium_date"))


In [0]:
# Get age and driver name for first win
first_win = df_results_age.join(
    first_win_date,
    on="driverId",
    how="inner"
).filter(
    col("race_date") == col("first_win_date")
).select(
    "driverId","forename",
    "surname","date","Age"
).withColumnRenamed("date", "first_win_date") \
 .withColumnRenamed("Age", "first_win_age")

# Show first win age for each driver
display(first_win.orderBy("first_win_age"))

#### Code Explain:
The code begins by identifying the first podium date for each driver. Filtering the `df_results_age` table, we keep only the rows where `positionOrder` is equal to `"1"`, `"2"`, or `"3"`. These values represent podium finishes.`groupBy("driverId")` groups the data by driver. `agg(min("race_date").alias("first_podium_date"))` finds the earliest race date on which each driver achieved a podium finish. `min()` is used because the first podium corresponds to the earliest date. The same logic is then applied to identify each driver’s first win. In this case, we keep only rows where `positionOrder` is equal to `"1"`, which represents a win.

After finding the first podium date for each driver, the code joins this summary table back to the full `df_results_age` DataFrame using `join(..., on="driverId", how="inner")` because the summary table only contains the driver ID and the first podium date.

`.filter(col("race_date") == col("first_podium_date"))` keeps only the row where the race date matches the driver’s first podium date. This ensures that the output contains the correct race where the first podium occurred.
The same process is repeated for first wins. The code joins `first_win_date` back to `df_results_age`, filters the rows where `race_date` matches `first_win_date`, and then selects the relevant columns.